In [0]:
from delta.tables import DeltaTable
import os
import sys

# Ensure Python can find utils.py from the parent directory
project_dir = "/Workspace/Users/ranayuvrajsingh1111@gmail.com/Olist ecomerce ETL"
if project_dir not in sys.path:
  sys.path.append(project_dir)

import utils
from pyspark.sql import functions as F

# ==========================================
# 1. EXTRACT GOLD FACT SALES
# ==========================================
print("Extracting gold_fact_sales for Customer LTV analytics...")
df_fact = spark.read.table("ecommerce_lakehouse.gold.gold_fact_sales")

# ==========================================
# 2. AGGREGATE METRICS BY CUSTOMER
# ==========================================
df_customer_summary = df_fact.groupBy("customer_key").agg(
    F.countDistinct("order_id").alias("total_orders_placed"),
    F.count("order_item_id").alias("total_items_purchased"),
    F.sum("total_item_amount").alias("lifetime_spend"),
    F.sum("freight_value").alias("total_freight_paid"),
    F.min("purchase_ts").alias("first_purchase_ts"),
    F.max("purchase_ts").alias("last_purchase_ts"),
    F.first("primary_payment_type").alias("preferred_payment_type"),
    F.max(F.when(F.col("is_delayed") == True, 1).otherwise(0)).alias(
        "has_experienced_delay"
    ),
)

# Derive tenure and average order value (AOV) metrics
df_transformed = (
    df_customer_summary.withColumn(
        "avg_order_value", F.col("lifetime_spend") / F.col("total_orders_placed")
    )
    .withColumn(
        "customer_tenure_days",
        F.datediff(F.col("last_purchase_ts"), F.col("first_purchase_ts")),
    )
    .orderBy(F.col("lifetime_spend").desc())
)

# ==========================================
# 3. WRITE TO GOLD LAYER
# ==========================================
target_table_name = "ecommerce_lakehouse.gold.gold_customer_ltv"
print(f"Writing data to Gold table: {target_table_name}")

df_transformed.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable(target_table_name)

print("Successfully generated gold_customer_ltv!")

Extracting gold_fact_sales for Customer LTV analytics...
Writing data to Gold table: ecommerce_lakehouse.gold.gold_customer_ltv
Successfully generated gold_customer_ltv!
